# Baseline Models (DummyRegressor)

This notebook builds **mean** and **median** dummy baselines and evaluates them on a **within-make split**:
for each `make`, we hold out ~20% of that make’s **models** for testing.

Why this matters: you evaluate generalization **within the same brand (make)** without making the test set a completely different set of brands.

In [50]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.dummy import DummyRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

def find_ml_root_for_import(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for candidate in (start, *start.parents):
        if (candidate / "app" / "core" / "config.py").exists():
            return candidate
    raise FileNotFoundError("Could not locate ml-service root containing app/core/config.py")

ML_ROOT = find_ml_root_for_import()
if str(ML_ROOT) not in sys.path:
    sys.path.insert(0, str(ML_ROOT))

print("ML_ROOT:", ML_ROOT)

from app.core.config import settings

ML_ROOT: /home/mo-seif/Documents/GP/ml-service


## 1) Load Data 

In [51]:

df = settings.load_data("processed")
df.head()

,make,model,year,transmission,fuel,mileage_km,location,engine_cc,horsepower,body_type,drivetrain,seating_capacity,brand_origin,car_segment,price_egp,price_egp_log
0,Deepal,S 07,2025,Automatic,hybrid,5000,October & Zayed,1500,318,SUV,AWD,5,chinese,suv,1650000,6.217484
1,BYD,F3,2024,Manual,petrol,4410,Cairo,1500,109,Sedan,FWD,5,chinese,family,630000,5.799341
2,BYD,F3,2025,Manual,petrol,33033,Nasr City,1500,109,Sedan,FWD,5,chinese,family,610000,5.785330
3,Subaru,Impreza,2009,Manual,petrol,98000,Maadi,1498,107,Sedan,AWD,5,japanese,family,410000,5.612784
4,BYD,F3,2024,Manual,petrol,78642,Cairo,1500,109,Sedan,FWD,5,chinese,family,570000,5.755875


In [52]:
df.dtypes.to_frame("dtype")

,dtype
make,str
model,str
year,int64
transmission,string
fuel,string
mileage_km,int64
location,str
engine_cc,int64
horsepower,int64
body_type,str


## 2) Train/Test Split Options (Choose Based on What You Want to Prove)

There are **two valid split modes** depending on your evaluation goal:

1) **`within_make_by_model`** (harder): for each `make`, hold out ~20% of that make's **models** for testing.
   - Tests: *generalization to unseen models within a known make*
   - Note: a **make+model mean/median baseline** cannot work here (those models are unseen).

2) **`within_make_model_by_rows`** (easier but practical): for each `(make, model)`, hold out ~20% of that group's **rows** for testing.
   - Tests: *generalization to new listings within the same make+model*
   - This is the correct split to evaluate a **make+model mean/median baseline**.

In [53]:
RANDOM_STATE = 42
TEST_SIZE = 0.2

# Choose split mode:
# - 'within_make_by_model'      : hold out models within each make (harder)
# - 'within_make_model_by_rows' : hold out rows within each (make, model) group (supports make+model baseline)
SPLIT_MODE = 'within_make_model_by_rows'

# Guardrails to avoid tiny unstable splits
MIN_MODELS_TO_SPLIT = 2  # only used in within_make_by_model
MIN_ROWS_PER_GROUP = 5   # only used in within_make_model_by_rows
FORCE_AT_LEAST_ONE_TEST_ITEM_PER_GROUP = True

df = df.copy()
df["make"] = df["make"].astype(str).str.strip()
df["model"] = df["model"].astype(str).str.strip()

rng = np.random.default_rng(RANDOM_STATE)

train_indices: list[int] = []
test_indices: list[int] = []

if SPLIT_MODE == 'within_make_by_model':
    # For each make: sample ~20% of models into test; train sees the other models of that make.
    for make, make_df in df.groupby("make", sort=False):
        unique_models = make_df["model"].dropna().unique()
        n_models = len(unique_models)
        if n_models < MIN_MODELS_TO_SPLIT:
            train_indices.extend(make_df.index.to_list())
            continue

        n_test_models = int(np.floor(TEST_SIZE * n_models))
        if FORCE_AT_LEAST_ONE_TEST_ITEM_PER_GROUP:
            n_test_models = max(1, n_test_models)
        n_test_models = min(n_test_models, n_models - 1)  # keep at least 1 model in train
        test_models = set(rng.choice(unique_models, size=n_test_models, replace=False).tolist())

        is_test = make_df["model"].isin(test_models)
        test_indices.extend(make_df.index[is_test].to_list())
        train_indices.extend(make_df.index[~is_test].to_list())

    train_df = df.loc[train_indices].reset_index(drop=True)
    test_df = df.loc[test_indices].reset_index(drop=True)

    print("Split mode:", SPLIT_MODE)
    print("Train rows:", len(train_df), " Test rows:", len(test_df))

    # Sanity checks
    assert set(train_indices).isdisjoint(set(test_indices))
    assert set(test_df["make"]).issubset(set(train_df["make"]))

    # Leakage check: within each make, no model overlap between train/test
    leaks = []
    for make in sorted(set(test_df["make"])):
        train_models = set(train_df.loc[train_df["make"] == make, "model"].unique())
        test_models = set(test_df.loc[test_df["make"] == make, "model"].unique())
        overlap = train_models.intersection(test_models)
        if overlap:
            leaks.append((make, len(overlap)))
    if leaks:
        raise AssertionError(f"Model leakage within make detected: {leaks[:10]}")
    print("Leakage check: OK (no model overlap within each make)")

elif SPLIT_MODE == 'within_make_model_by_rows':
    # For each (make, model): sample ~20% of rows into test; train sees same make+model but different rows.
    for (make, model), gdf in df.groupby(["make", "model"], sort=False):
        n = len(gdf)
        if n < MIN_ROWS_PER_GROUP:
            # keep small groups in train only
            train_indices.extend(gdf.index.to_list())
            continue

        n_test_rows = int(np.floor(TEST_SIZE * n))
        if FORCE_AT_LEAST_ONE_TEST_ITEM_PER_GROUP:
            n_test_rows = max(1, n_test_rows)
        n_test_rows = min(n_test_rows, n - 1)  # keep at least 1 row in train
        chosen = rng.choice(gdf.index.to_numpy(), size=n_test_rows, replace=False)
        chosen_set = set(chosen.tolist())

        test_indices.extend(list(chosen_set))
        train_indices.extend([idx for idx in gdf.index.to_list() if idx not in chosen_set])

    train_df = df.loc[train_indices].reset_index(drop=True)
    test_df = df.loc[test_indices].reset_index(drop=True)

    print("Split mode:", SPLIT_MODE)
    print("Train rows:", len(train_df), " Test rows:", len(test_df))

    # Sanity checks
    assert set(train_indices).isdisjoint(set(test_indices))
    assert set(test_df["make"]).issubset(set(train_df["make"]))

    # Leakage check: groups should overlap (by design), but rows must be disjoint (asserted above).
    overlap_groups = set((train_df["make"] + " | " + train_df["model"]).unique()).intersection(
        set((test_df["make"] + " | " + test_df["model"]).unique())
    )
    print("Groups present in both train and test:", len(overlap_groups))

else:
    raise ValueError("SPLIT_MODE must be 'within_make_by_model' or 'within_make_model_by_rows'")

Split mode: within_make_model_by_rows
Train rows: 18812  Test rows: 4440
Groups present in both train and test: 570


## 3) Define Metrics

We evaluate on both:
- `price_egp` (business-readable)
- `price_egp_log` (often easier for models, but less interpretable)

In [54]:
def safe_mape(y_true: np.ndarray, y_pred: np.ndarray, eps: float = 1.0) -> float:
    denom = np.clip(np.abs(y_true), eps, None)
    return float(np.mean(np.abs((y_true - y_pred) / denom)) * 100.0)

def regression_report(y_true: np.ndarray, y_pred: np.ndarray) -> dict:
    mse = float(mean_squared_error(y_true, y_pred))
    rmse = float(np.sqrt(mse))
    return {
        'MAE': float(mean_absolute_error(y_true, y_pred)),
        'MSE': mse,
        'RMSE': rmse,
        'MAPE_%': safe_mape(y_true, y_pred),
        'R2': float(r2_score(y_true, y_pred)),
    }

## 4) Train + Evaluate Dummy Baselines

Important: DummyRegressor **does not use features**. We pass a constant `X` just to satisfy the sklearn API.

In [55]:
X_train = np.zeros((len(train_df), 1))
X_test = np.zeros((len(test_df), 1))

y_train_egp = train_df['price_egp'].to_numpy(dtype=float)
y_test_egp = test_df['price_egp'].to_numpy(dtype=float)

y_train_log = train_df['price_egp_log'].to_numpy(dtype=float)
y_test_log = test_df['price_egp_log'].to_numpy(dtype=float)

In [56]:
def _group_stats(train_df: pd.DataFrame, group_cols: list[str], target_col: str, agg: str) -> pd.DataFrame:
    if agg == 'mean':
        out = train_df.groupby(group_cols, as_index=False)[target_col].mean()
    elif agg == 'median':
        out = train_df.groupby(group_cols, as_index=False)[target_col].median()
    else:
        raise ValueError("agg must be 'mean' or 'median'")
    return out.rename(columns={target_col: 'pred'})

def predict_make_model_cascade(
    train_df: pd.DataFrame,
    test_df: pd.DataFrame,
    *,
    target_col: str,
    agg: str,
    ) -> tuple[np.ndarray, dict]:
    """
    Predict using (make, model) aggregate, fallback to make aggregate, fallback to global aggregate.
    Returns predictions + coverage stats.
    """
    stats_mm = _group_stats(train_df, ['make', 'model'], target_col, agg)
    stats_make = _group_stats(train_df, ['make'], target_col, agg)
    global_value = float(train_df[target_col].mean() if agg == 'mean' else train_df[target_col].median())

    merged = test_df.merge(stats_mm, on=['make', 'model'], how='left')
    coverage_mm = float(merged['pred'].notna().mean())

    missing_mm = merged['pred'].isna()
    if missing_mm.any():
        merged = merged.merge(stats_make.rename(columns={'pred': 'pred_make'}), on=['make'], how='left')
        merged.loc[missing_mm, 'pred'] = merged.loc[missing_mm, 'pred_make']
    coverage_after_make = float(merged['pred'].notna().mean())

    merged['pred'] = merged['pred'].fillna(global_value)
    pred = merged['pred'].to_numpy(dtype=float)

    return pred

def eval_group_make(train_df: pd.DataFrame, test_df: pd.DataFrame, *, target_col: str, agg: str, y_test: np.ndarray) -> dict:
    stats_make = _group_stats(train_df, ['make'], target_col, agg)
    global_value = float(train_df[target_col].mean() if agg == 'mean' else train_df[target_col].median())
    merged = test_df.merge(stats_make, on=['make'], how='left')
    coverage = float(merged['pred'].notna().mean())
    pred = merged['pred'].fillna(global_value).to_numpy(dtype=float)
    return {
        'baseline': f'group_make_{agg} (fallback global_{agg})',
        **regression_report(y_test, pred),
    }

def eval_group_make_model_cascade(train_df: pd.DataFrame, test_df: pd.DataFrame, *, target_col: str, agg: str, y_test: np.ndarray) -> dict:
    pred = predict_make_model_cascade(train_df, test_df, target_col=target_col, agg=agg)
    return {
        'baseline': f'group_make_model_{agg} (fallback make_{agg} → global_{agg})',
        **regression_report(y_test, pred),
    }

# ---- Evaluate baselines on price_egp ----
baselines_price = [
    eval_group_make(train_df, test_df, target_col='price_egp', agg='mean', y_test=y_test_egp),
    eval_group_make(train_df, test_df, target_col='price_egp', agg='median', y_test=y_test_egp),
    eval_group_make_model_cascade(train_df, test_df, target_col='price_egp', agg='mean', y_test=y_test_egp),
    eval_group_make_model_cascade(train_df, test_df, target_col='price_egp', agg='median', y_test=y_test_egp),
 ]

results_price = pd.DataFrame(baselines_price).sort_values('MAE')
print('=== Baselines on price_egp (sorted by MAE) ===')
results_price

=== Baselines on price_egp (sorted by MAE) ===


,baseline,MAE,MSE,RMSE,MAPE_%,R2
3,group_make_model_median (fallback make_median ...,316670.480856,6.579245e+11,8.111255e+05,37.276887,0.739570
2,group_make_model_mean (fallback make_mean → gl...,323448.374474,6.261495e+11,7.912961e+05,41.538387,0.752148
1,group_make_median (fallback global_median),528129.156306,1.374812e+12,1.172524e+06,63.193455,0.455800
0,group_make_mean (fallback global_mean),562337.329884,1.294426e+12,1.137728e+06,86.767237,0.487620


In [57]:
# Auto assessment: pick best baseline by MAE
best = results_price.sort_values("MAE").iloc[0]

print("Best baseline (price_egp):", best["baseline"])
print("  MAE:", f"{best['MAE']:,.0f}")
print("  MAPE%:", f"{best['MAPE_%']:.2f}")
print("  MSE:", f"{best['MSE']:.3e}")
print("  R2:", f"{best['R2']:.4f}")

print("\nRecommendation:")
print("- This is your strongest non-ML baseline for the chosen SPLIT_MODE.")
print("- Next step: train LightGBM median model and beat this MAE/MAPE.")
print("- Use SPLIT_MODE='within_make_by_model' if you care about unseen models; compare against the make-level baselines.")

Best baseline (price_egp): group_make_model_median (fallback make_median → global_median)
  MAE: 316,670
  MAPE%: 37.28
  MSE: 6.579e+11
  R2: 0.7396

Recommendation:
- This is your strongest non-ML baseline for the chosen SPLIT_MODE.
- Next step: train LightGBM median model and beat this MAE/MAPE.
- Use SPLIT_MODE='within_make_by_model' if you care about unseen models; compare against the make-level baselines.


## Final Assessment (What This Baseline Means)

You were right: **group mean/median baselines** (like `make+model` median) are much stronger than a global Dummy baseline, and they often avoid the negative $R^2$ issue.

Key point: the baseline must match the split you’re using:

- If `SPLIT_MODE = 'within_make_model_by_rows'`:
  - `make+model` mean/median is a fair and strong baseline (because the same group exists in train).

- If `SPLIT_MODE = 'within_make_by_model'`:
  - `make+model` mean/median cannot work directly (those models are held out).
  - In that case, compare against `make` mean/median baselines (or use the cascade baseline which falls back to make/global).

**Recommendation:** use the best group baseline as your *non-ML benchmark*, then train LightGBM and require a clear improvement before building the full pipeline.